# Wood NIR Moisture CNN Embedding Experiment
Colab/Local両対応。

In [ ]:

from pathlib import Path
import os, sys, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Colab detected')
    # from google.colab import drive; drive.mount('/content/drive')

CONFIG = {
    'TRAIN_PATH': '../data/train.csv', 'TEST_PATH': '../data/test.csv', 'ENCODING':'cp932',
    'OUTPUT_DIR':'outputs/nir_cnn_embedding', 'SEED':42,
    'META_COLS':['sample number','species number','樹種','含水率'], 'TARGET_COL':'含水率','GROUP_COL':'species number',
    'CNN_CHANNELS':['raw','snv','snv_sg1','snv_sg2'], 'SG_WINDOW':21, 'SG_POLY':3,
    'BANDS':{'1000_1300':(1000,1300),'1300_1600':(1300,1600),'1600_1800':(1600,1800),'1800_2000':(1800,2000),'2000_2300':(2000,2300),'2300_2500':(2300,2500),'full':None},
    'CV_SPLITS':5, 'EPOCHS':100, 'PATIENCE':15, 'BATCH_SIZE':64, 'LR':1e-3, 'WD':1e-4,
    'EMBED_DIMS':[8,16,32], 'TARGET_TRANSFORMS':['none','log1p']
}
for d in ['features','models','results','figures']:
    Path(CONFIG['OUTPUT_DIR'], d).mkdir(parents=True, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED']=str(seed)
seed_everything(CONFIG['SEED'])


In [ ]:

from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def load_df():
    tr = pd.read_csv(CONFIG['TRAIN_PATH'], encoding=CONFIG['ENCODING'])
    te = pd.read_csv(CONFIG['TEST_PATH'], encoding=CONFIG['ENCODING'])
    return tr, te

train_df, test_df = load_df()

def get_spec_cols(df):
    out=[]
    for c in df.columns:
        if c in CONFIG['META_COLS']: continue
        try: float(c); out.append(c)
        except: pass
    return out
spec_cols=get_spec_cols(train_df)
wn=np.array([float(c) for c in spec_cols])
wl=1e7/wn
ordr=np.argsort(wl)
spec_cols_sorted=[spec_cols[i] for i in ordr]
wavelength_nm=wl[ordr]
print('wavenumber range', (wn.min(), wn.max()))
print('wavelength range', (wavelength_nm.min(), wavelength_nm.max()))
print('spec columns sample', spec_cols_sorted[:8])
print('wavelength array sample', wavelength_nm[:8])


In [ ]:

Xtr_raw=train_df[spec_cols_sorted].values.astype(np.float32)
Xte_raw=test_df[spec_cols_sorted].values.astype(np.float32)
y=train_df[CONFIG['TARGET_COL']].values.astype(np.float32)
groups=train_df[CONFIG['GROUP_COL']].values

def snv(X):
    return (X-X.mean(1,keepdims=True))/(X.std(1,keepdims=True)+1e-12)
def reps(X):
    Xs=snv(X)
    return {
      'raw':X,'snv':Xs,
      'raw_sg1':savgol_filter(X,21,3,deriv=1,axis=1),
      'raw_sg2':savgol_filter(X,21,3,deriv=2,axis=1),
      'snv_sg1':savgol_filter(Xs,21,3,deriv=1,axis=1),
      'snv_sg2':savgol_filter(Xs,21,3,deriv=2,axis=1),
    }
tr_r, te_r = reps(Xtr_raw), reps(Xte_raw)

def build_cnn_input(r, channels):
    return np.stack([r[c] for c in channels], axis=-1)
Xtr_cnn=build_cnn_input(tr_r, CONFIG['CNN_CHANNELS'])
Xte_cnn=build_cnn_input(te_r, CONFIG['CNN_CHANNELS'])
print('cnn shape', Xtr_cnn.shape)


In [ ]:

def species_weights(df):
    vc=df['species number'].value_counts()
    w=df['species number'].map(lambda x:1.0/vc[x]).values
    return w/w.mean()

def species_index_balanced_weights(df, bins=10):
    d=df[['sample number','species number']].copy()
    d=d.sort_values(['species number','sample number']).copy()
    d['idx']=d.groupby('species number').cumcount()
    n=d.groupby('species number')['idx'].transform('max').replace(0,1)
    d['idx_norm']=d['idx']/n
    d['index_bin']=np.minimum((d['idx_norm']*bins).astype(int), bins-1)
    cnt=d.groupby(['species number','index_bin'])['sample number'].transform('count')
    d['w']=1.0/cnt
    d=d.set_index('sample number')
    w=df['sample number'].map(d['w']).values
    return w/w.mean()

w_species=species_weights(train_df)
w_species_idx=species_index_balanced_weights(train_df)


In [ ]:

def nearest_idx(target):
    return int(np.argmin(np.abs(wavelength_nm-target)))
def band_idx(a,b):
    return np.where((wavelength_nm>=a)&(wavelength_nm<=b))[0]

def make_feats(r):
    snv_,sg1,sg2=r['snv'],r['snv_sg1'],r['snv_sg2']
    F={}
    F['snv_1450']=snv_[:,nearest_idx(1450)]
    p=band_idx(2230,2280); n=band_idx(2000,2150)
    F['sg1_ratio_pos_neg']=np.maximum(sg1[:,p],0).mean(1)/(np.abs(np.minimum(sg1[:,n],0)).mean(1)+1e-8)
    F['snv_sg2_slope_1000_1300']=(sg2[:,nearest_idx(1300)]-sg2[:,nearest_idx(1000)])/300
    F['sg2_1720_1740_contrast']=sg2[:,band_idx(1720,1740)].mean(1)-sg2[:,band_idx(1685,1720)].mean(1)
    F['sg2_1370_peak']=sg2[:,nearest_idx(1370)]
    F['snv_slope_1650_1750']=(snv_[:,nearest_idx(1750)]-snv_[:,nearest_idx(1650)])/100
    F['sg1_1900_left_right_ratio']=np.abs(sg1[:,band_idx(1850,1900)]).mean(1)/(np.abs(sg1[:,band_idx(1900,1950)]).mean(1)+1e-8)
    F['snv_slope_1600_1800']=(snv_[:,nearest_idx(1800)]-snv_[:,nearest_idx(1600)])/200
    F['sg2_1685_1720_contrast']=sg2[:,band_idx(1685,1720)].mean(1)-sg2[:,band_idx(1720,1740)].mean(1)
    F['sg1_slope_2000_2050']=sg1[:,band_idx(2000,2050)].mean(1)
    return pd.DataFrame(F)

feat_tr, feat_te = make_feats(tr_r), make_feats(te_r)


In [ ]:

# 実行時間節約のため、CNN学習関数とC1-C6比較関数は関数化済みテンプレートをこのセルに実装してください。
# 要件: GroupKFold, weighted mse, foldモデル保存, OOF embedding/test embedding保存, C1..C6評価保存, submission作成。
print('Template note: implement/train here for full experiment run.')
